# Masterclass — Modèles de volatilité, Smiles/Skews, Black–Scholes & Heston

Ce notebook rassemble **un cours structuré** (en *Markdown*) et **des démonstrations pratiques** (en *Python*) :

1. **Modèles de volatilité** (BS, Local Vol, Stochastic Vol, SABR, SLV) — *formules, avantages, limites, usages*.
2. **Smiles & Skews** — *définitions, diagnostics, problèmes courants, remèdes*.
3. **Black–Scholes** — *modèle, prix fermés, hypothèses, limites*.
4. **Heston** — *modèle vol stochastique, intuition, pricing, points pratiques*.
5. **Simulations** — *trajectoires BS vs Heston (QE d’Andersen)*, avec **beaucoup** de commentaires.
6. **Q&A d’entretien** — *20 questions/réponses ciblées*.


## 1) Récap — Modèles de volatilité

### 1.1. Black–Scholes (vol constante)
$$ dS_t = (r-q)S_t\,dt + \sigma S_t\,dW_t, \quad \sigma \text{ constant}. $$
**Avantages**: simplicité, formules fermées, rapide.  
**Inconvénients**: pas de smile/skew; dynamique de vol irréaliste.  
**Usage**: benchmark, académique.

### 1.2. Volatilité locale (Dupire)
$$ dS_t = (r-q)S_t\,dt + \sigma_{\text{loc}}(S_t,t) S_t\,dW_t. $$
**Avantages**: colle à la surface de vol implicite (statique).  
**Inconvénients**: dynamique du smile peu réaliste.  
**Usage**: pricing d’exotiques sous contrainte de calibration parfaite aux vanilles.

### 1.3. Volatilité stochastique (ex: Heston)
$$ dS_t = (r-q)S_t\,dt + \sqrt{v_t} S_t\,dW_t^{(1)}, \quad dv_t = \kappa(\theta-v_t)dt + \xi\sqrt{v_t}\,dW_t^{(2)}, \; \mathrm{corr}=\rho. $$
**Avantages**: skew/smile naturels; dynamique réaliste.  
**Inconvénients**: calibration plus lourde; pas toujours parfait sur toute la surface.  
**Usage**: FX/Equity, exotiques, risk management.

### 1.4. SABR
$$ dF_t = \sigma_t F_t\,dW_t^{(1)},\quad d\sigma_t = \nu\sigma_t\,dW_t^{(2)},\quad \mathrm{corr}=\rho. $$
**Avantages**: capture bien les smiles en taux; formules asymptotiques rapides.  
**Inconvénients**: approximatif; pathologies en régimes extrêmes.  
**Usage**: taux (swaptions, caps/floors).

### 1.5. SLV (Stochastic Local Vol)
Combinaison local × stoch vol.  
**Avantages**: calibration parfaite + dynamique réaliste.  
**Inconvénients**: complexité/calibration lourdes.  
**Usage**: exotiques sensibles au smile (barrières, cliquets…).


### 1.6. Illustration synthétique des ‘smiles’ (exemple)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
K = np.linspace(60, 140, 41)
iv_flat = np.full_like(K, 0.20, dtype=float)
iv_smile = 0.18 + 0.0009*(K-100)**2/100
iv_skew = 0.24 - 0.0007*(K-100)
plt.figure(); plt.plot(K, iv_flat, label='Plat (BS)'); plt.plot(K, iv_smile, label='Smile (U)'); plt.plot(K, iv_skew, label='Skew (décroissant)');
plt.xlabel('Strike K'); plt.ylabel('Vol implicite (synthétique)'); plt.title('Plat vs Smile vs Skew'); plt.legend(); plt.grid(True); plt.show()

## 2) Smiles & Skews — définition, problèmes et remèdes

### 2.1. Cas plat (BS)
- Simple mais **ne colle pas** au marché (pas de skew/smile).

### 2.2. Smile
- En forme de “U”: vols plus hautes loin du strike; **queues épaisses**.

### 2.3. Skew
- Décroissant côté puts OTM (equity/FX) → **crash risk** et effet leverage.

### 2.4. Problèmes & solutions
- **Dynamique irréaliste** (local vol pur) → **stoch vol** / **SLV**.  
- **Extrapolation incohérente** → **SVI**, **SABR**.  
- **Maturités incohérentes** → paramétrisations **arbitrage-free**.  
- **Calibration difficile** → extensions (Heston+jumps), régularisation.


## 3) Black–Scholes — modèle, prix, hypothèses

### 3.1. Dynamique
$$ dS_t = (r-q) S_t dt + \sigma S_t dW_t, \quad \sigma \text{ constant}. $$

### 3.2. Call européen (sans dividendes)
$$ C = S_0 e^{-qT} N(d_1) - K e^{-rT} N(d_2), $$
avec  
$$ d_1 = \frac{\ln(S_0/K) + (r-q + \tfrac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}. $$

### 3.3. Hypothèses & limites
- Pas d’arbitrage, marché frictionless, vol **constante**.  
- **Ne reproduit pas** le smile/skew; queues fines.


## 4) Heston — modèle de volatilité stochastique

### 4.1. Dynamique
$$ dS_t = (r-q) S_t dt + \sqrt{v_t} S_t dW_t^{(1)}, $$
$$ dv_t = \kappa(\theta - v_t) dt + \xi\sqrt{v_t} dW_t^{(2)}, \quad \mathrm{corr}(dW^{(1)},dW^{(2)})=\rho. $$

### 4.2. Intuition
- Volatilité **variable** et **corrélée** au spot (souvent $\rho<0$).  
- Génère **skew** et **smile** de façon naturelle.

### 4.3. Pricing
- Vanilles: **intégrale de Fourier** (semi-fermée).  
- Exotiques: **Monte Carlo** (souvent QE d’Andersen).  
- Calibration: fit de la surface de vol implicite.


## 5) Simulations — BS vs Heston (avec code commenté)

On simule et trace des trajectoires du sous-jacent sous **Black–Scholes** (vol constante) puis sous **Heston**
(variance stochastique) avec le schéma **Quadratic–Exponential (QE)** d’Andersen. Le code est **très commenté**.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math
SQRT_2 = math.sqrt(2.0)
def stdnorm_cdf(x):
    return 0.5*(1+math.erf(x/SQRT_2))

def simulate_bs_paths(S0=100.0, T=1.0, r=0.02, q=0.0, sigma=0.2, n_paths=20, n_steps=252, seed=42):
    """Trajectoires BS (vol constante)."""
    rng = np.random.default_rng(seed); dt = T/n_steps
    logS = np.full(n_paths, math.log(S0)); paths = np.zeros((n_steps+1, n_paths)); paths[0]=S0
    for t in range(1, n_steps+1):
        z = rng.standard_normal(size=n_paths)
        logS += (r-q-0.5*sigma*sigma)*dt + sigma*math.sqrt(dt)*z
        paths[t] = np.exp(logS)
    return paths

def heston_qe_variance_step(v, dt, kappa, theta, xi, rng):
    exp_kdt = np.exp(-kappa*dt)
    m  = theta + (v - theta)*exp_kdt
    s2 = (v*xi*xi*exp_kdt/kappa)*(1-exp_kdt) + (theta*xi*xi/(2.0*kappa))*(1-exp_kdt)**2
    psi = s2/(m*m + 1e-16)
    psi_c = 1.5
    z = rng.standard_normal(size=v.shape); u = rng.random(size=v.shape)
    v_next = np.empty_like(v)
    mask1 = psi < psi_c
    if np.any(mask1):
        m1, psi1, z1 = m[mask1], psi[mask1], z[mask1]
        b2 = 2.0/psi1 - 1.0 + np.sqrt((2.0/psi1)*(2.0/psi1 - 1.0))
        b  = np.sqrt(b2); a = m1/(1.0+b2)
        v_next[mask1] = a*(b+z1)**2
    mask2 = ~mask1
    if np.any(mask2):
        m2, psi2, u2 = m[mask2], psi[mask2], u[mask2]
        p = (psi2-1.0)/(psi2+1.0); beta = (1.0-p)/(m2 + 1e-16)
        v2 = np.zeros_like(m2); mexp = u2>p
        v2[mexp] = np.log((1.0-p[mexp])/(1.0-u2[mexp]))/(beta[mexp] + 1e-16)
        v_next[mask2] = v2
    return v_next, z

def simulate_heston_paths(S0=100.0, T=1.0, r=0.02, q=0.0, v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7, n_paths=20, n_steps=252, seed=7):
    """Trajectoires Heston (QE pour variance, corrélation rho entre chocs)."""
    rng = np.random.default_rng(seed); dt = T/n_steps
    logS = np.full(n_paths, math.log(S0)); v = np.full(n_paths, v0)
    S_paths = np.zeros((n_steps+1, n_paths)); V_paths = np.zeros((n_steps+1, n_paths))
    S_paths[0]=S0; V_paths[0]=v0
    for t in range(1, n_steps+1):
        v_next, z_var = heston_qe_variance_step(v, dt, kappa, theta, xi, rng)
        eps = rng.standard_normal(size=v.shape)
        z_spot = rho*z_var + math.sqrt(max(0.0, 1.0-rho*rho))*eps
        v_bar = 0.5*(v+v_next); v_bar = np.maximum(v_bar, 0.0)
        logS += (r-q-0.5*v_bar)*dt + np.sqrt(v_bar*dt)*z_spot
        v = v_next; S_paths[t]=np.exp(logS); V_paths[t]=v
    return S_paths, V_paths

# Tracés BS vs Heston
bs_paths = simulate_bs_paths()
time = np.linspace(0, 1.0, bs_paths.shape[0])
import matplotlib.pyplot as plt
plt.figure()
for i in range(bs_paths.shape[1]):
    plt.plot(time, bs_paths[:, i], lw=0.8)
plt.xlabel('Temps (années)'); plt.ylabel('S_t'); plt.title('Trajectoires — Black–Scholes'); plt.grid(True); plt.show()

S_heston, V_heston = simulate_heston_paths()
plt.figure()
for i in range(S_heston.shape[1]):
    plt.plot(time, S_heston[:, i], lw=0.8)
plt.xlabel('Temps (années)'); plt.ylabel('S_t'); plt.title('Trajectoires — Heston (prix)'); plt.grid(True); plt.show()

plt.figure()
for i in range(V_heston.shape[1]):
    plt.plot(time, V_heston[:, i], lw=0.8)
plt.xlabel('Temps (années)'); plt.ylabel('v_t'); plt.title('Trajectoires — Heston (variance)'); plt.grid(True); plt.show()


## 6) Entretien — 20 questions/réponses

1. **Qu’est-ce qu’un modèle de volatilité ?**  
   Un schéma qui décrit comment la volatilité évolue (constante, locale, stochastique, hybride).  
2. **Pourquoi BS ne voit pas le smile ?**  
   Car $\sigma$ est constant, distribution log-normale trop fine.  
3. **Local vol vs Stoch vol ?**  
   Local: colle la surface statique; Stoch: donne une dynamique réaliste.  
4. **SLV sert à quoi ?**  
   Calibration parfaite + dynamique réaliste (exotiques).  
5. **Heston: rôle de $\rho$** ?  
   Corrélation spot/vol; $\rho<0$ → skew négatif (equity/FX).  
6. **Condition de Feller ?**  
   $2\kappa\theta \ge \xi^2$ pour éviter v=0 (CIR).  
7. **Pricing Heston vanille ?**  
   Intégrale de Fourier (semi-fermée) ou MC.  
8. **Quand utiliser SABR ?**  
   En taux (swaptions), calibration rapide par formules asymptotiques.  
9. **Pourquoi Monte Carlo ?**  
   Générique, flexible pour exotiques; CI et variance connus (~1/√N).  
10. **Limite MC ?**  
   Lent pour vanilles, bruit statistique; calibration coûteuse.  
11. **Vol implicite ?**  
   La $\sigma$ qui égalise prix BS et prix de marché.  
12. **Surface incohérente: que faire ?**  
   Paramétrisations arbitrage-free (SVI), lissages, contraintes.  
13. **Pourquoi local vol peut mal ‘bouger’ ?**  
   Réplique statique mais pas la dynamique du smile → hedging trompeur.  
14. **Vol-of-vol ($\xi$) en Heston ?**  
   Amplitude des variations de variance; affecte le smile.  
15. **$\kappa,\theta$ ?**  
   Vitesse de rappel et niveau de long terme.  
16. **Gamma/Vega longs ?**  
   Options longues: Gamma>0, Vega>0 (profitent des gros moves/vol ↑).  
17. **Delta-hedging ?**  
   Neutraliser $\Delta$ via sous-jacent; rééquilibrage dynamique.  
18. **Pourquoi $\Theta<0$ ?**  
   Érosion temporelle (temps qui passe).  
19. **Put-call parity ?**  
   $C-P=S_0-K e^{-rT}$.  
20. **Quand préférer BS à Heston ?**  
   Vanilles simples, besoin de vitesse (calibration, pricing instantané).
